# Stage 1 model

In [1]:
import pandas as pd
import plotly.express as px

from rich import print

from ambdes import Model, Runner, SimConfig, ArrivalConfig, TimesConfig

## Run model for 100 minutes with log messages

TODO: Convert this into logging tests which compare patient results with record in the vidigi logger.

In [ ]:
class ArrivalConfig:
    """Prepare arrival inputs for the simulation.

    Attributes
    ----------
    arrival_df : pd.DataFrame
        Mean arrival counts by day of week and response category.
    variation_df : pd.DataFrame
        Summary of variation in category proportions across days of the week.
    category_proportions : pd.Series
        Mean proportion of arrivals in each response category across the week.
    nspp_df : pd.DataFrame
        Arrival schedule in the format required by `sim_tools` `NSPPThinning`.

    """

    def __init__(self, arrival_csv, model_type):
        """Initialise ArrivalConfig.

        Parameters
        ----------
        arrival_csv : str | Path
            Path to CSV containing arrival counts by day of week and response
            category.
        model_type : str
            Controls which columns are required and which derived outputs
            are produced.

        """
        if model_type not in ["aggregate", "cycle"]:
            raise ValueError("model_type must be aggregate or cycle.")

        self.arrival_df = pd.read_csv(arrival_csv)

        # Sum conveyed & non-conveyed
        if model_type == "cycle":
            counts = (
                self.arrival_df.groupby(
                    ["day", "category"], observed=False
                )["count"]
                .sum()
                .reset_index()
            )
        else:
            counts = self.arrival_df.copy()

        # Convert to wide format
        wide_df = (
            counts
            .pivot(index="day", columns="category", values="count")
            .reindex(["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"])
        )
        wide_df.columns.name = None

        # Convert to proportion by response category for each day
        proportion_df = wide_df.div(wide_df.sum(axis=1), axis=0)

        # Summarise the variations in proportions by day of week
        self.variation_df = pd.DataFrame(
            {
                "mean": proportion_df.mean(axis=0),
                "min": proportion_df.min(axis=0),
                "max": proportion_df.max(axis=0),
                "range": proportion_df.max(axis=0) - proportion_df.min(axis=0),
                "sd": proportion_df.std(axis=0),
            }
        )

        # Get overall mean proportion by response category
        self.category_proportions = proportion_df.mean(axis=0)

        # Get count of total arrivals per day
        arrivals_per_day = wide_df.sum(axis=1)

        # Convert to format required by sim_tools NSPPThinning class
        # It requires a dataframe with columns "t" (timepoint when arrival
        # rate changes) and "mean_iat" (mean inter-arrival time)
        self.nspp_df = pd.DataFrame(
            {
                "t": range(0, 7 * 1440, 1440),
                "mean_iat": 1440 / arrivals_per_day.values,
            }
        )

        if model_type=="cycle":
            

In [3]:
import pandas as pd

,area,day,category,conveyed,count
0,southwest,monday,C1,all,25
1,southwest,tuesday,C1,all,24
2,southwest,wednesday,C1,all,24
3,southwest,thursday,C1,all,24
4,southwest,friday,C1,all,25
5,southwest,saturday,C1,all,28
6,southwest,sunday,C1,all,27
7,southwest,monday,C2,all,310
8,southwest,tuesday,C2,all,295
9,southwest,wednesday,C2,all,295


In [73]:
arrival_config = ArrivalConfig(arrival_csv="data/arrivals_conveyed.csv", model_type="cycle")

In [70]:
arrival_config = ArrivalConfig(arrival_csv="data/arrivals.csv", model_type="aggregate")

In [ ]:
arrival_config.arrival_df.groupby(["day", "category"])

,day,category,conveyed,count
0,monday,C1,True,20
1,monday,C1,False,5
2,tuesday,C1,True,19
3,tuesday,C1,False,5
4,wednesday,C1,True,19
5,wednesday,C1,False,5
6,thursday,C1,True,19
7,thursday,C1,False,5
8,friday,C1,True,20
9,friday,C1,False,5


In [46]:
#data = pd.read_csv("data/arrivals.csv")
data = pd.read_csv("data/arrivals_conveyed.csv")

per_day_cat = (
    data.groupby(["day", "category"], observed=False)["count"]
    .sum()
    .reset_index()
)
wide_df = (
    per_day_cat
    .pivot(index="day", columns="category", values="count")
    .reindex(["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"])
)
wide_df.columns.name = None

In [47]:
data

,day,category,conveyed,count
0,monday,C1,True,20
1,monday,C1,False,5
2,tuesday,C1,True,19
3,tuesday,C1,False,5
4,wednesday,C1,True,19
5,wednesday,C1,False,5
6,thursday,C1,True,19
7,thursday,C1,False,5
8,friday,C1,True,20
9,friday,C1,False,5


In [39]:
arrival_config = ArrivalConfig(arrival_csv="data/arrivals_conveyed.csv")

KeyError: 'day'

In [29]:
arrival_config = ArrivalConfig(arrival_csv="data/arrivals.csv")
display(arrival_config.arrival_df)
display(arrival_config.nspp_df)

KeyError: 'day'

In [ ]:
long_df = (
    arrival_config.arrival_df
    .reset_index()                             # bring day index into a column
    .melt(
        id_vars="index",                       # the day column created by reset_index()
        var_name="category",                  # C1..C4
        value_name="count",                   # counts
    )
    .rename(columns={"index": "day"})
)
long_df

In [ ]:
long_df.to_csv("data/arrivals.csv", index=False)

In [ ]:
times_config = TimesConfig(times_csv="data/times.csv")

Inspect this to see if times actually vary by response category:

In [ ]:
times_config.times_df

In [ ]:
times_config.lognormal_config("travel_to_scene")

In [ ]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config
)
config.n_ambulances = 1

In [ ]:
print(config.dist_config)

In [ ]:
model = Model(run_number=0, config=config)
print(model.dists)

In [ ]:
model.dists["time_to_scene"]["C1"].sample()

In [ ]:
model.run()

In [ ]:
log = model.logger.to_dataframe()

In [ ]:
print(model.patients[0].__dict__)
log[log["entity_id"] == 1]

In [ ]:
print(model.patients[1].__dict__)
log[log["entity_id"] == 2]

## Run model for longer and inspect patient times

In [ ]:
config = SimConfig(
    arrival_config=arrival_config,
    times_config=times_config,
    warm_up_period=0,
    data_collection_period=10080,  # One week
)
model = Model(run_number=0, config=config)
model.run()

In [ ]:
df = pd.DataFrame(
    {
        "response_time": [p.response_time for p in model.patients],
        "category": [p.category for p in model.patients],
    }
)

fig = px.histogram(
    df,
    x="response_time",
    facet_col="category",
    nbins=50,
    category_orders={"category": ["C1", "C2", "C3", "C4"]},
    labels={
        "response_time": "Response time (minutes)",
        "category": "Category",
    },
    title="Distribution of response times by category",
)

# fig.update_yaxes(
#    matches=None,
#    showticklabels=True,
# )
fig.layout.yaxis.title.text = "Number of patients"

fig.show()

In [ ]:
for cat in ["C1", "C2", "C3", "C4"]:
    fig = px.histogram(
        df[df["category"] == cat],
        x="response_time",
        nbins=20,
        title=f"Response times: {cat}",
        labels={"response_time": "Response time (minutes)"},
    )
    fig.update_yaxes(title_text="Number of patients")
    fig.show()

## Average results

In [ ]:
runner = Runner(config)

In [ ]:
results = runner.run_reps()

In [ ]:
results["patients"]

In [ ]:
results["run"]

In [ ]:
results["overall"]

In [ ]:
config.n_ambulances = 1
config.data_collection_period = 50_000
config.log_to_console = False
runner = Runner(config)
results = runner.run_single(run_number=0)

In [ ]:
results["run"]

In [ ]:
results["patients"].head(20)

In [ ]:
results["model"].logger.to_dataframe().head(30)